# Trace Count v4: v2-Style Steering

This notebook follows `pipeline_v4_steering_codex_prompt.md`. It intentionally reuses the v2 symbolic counting setup and v2-style HuggingFace GPT-2 architecture with learned absolute positional embeddings. The new part is mechanistic: hidden-state cache, probes, direction extraction, steering, and patching.

In [ ]:
from pathlib import Path
import importlib
import os, subprocess, sys

REPAIR_NUMPY_ABI = True  # If pandas/numpy ABI is broken, repair once and ask for kernel restart.
INSTALL_MINIMAL_DEPS = False
INSTALL_EDITABLE_PACKAGE = False  # Usually unnecessary from repo root; if enabled, uses --no-deps.

REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
if Path('/content').exists():
    repo_dir = Path('/content/Synthetic_CoT_NiaH_Count')
    if repo_dir.exists():
        os.chdir(repo_dir)
        subprocess.run(['git', 'pull'], check=False)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(repo_dir)], check=True)
        os.chdir(repo_dir)



def ensure_science_stack_abi():
    """Colab can break pandas if numpy is upgraded/downgraded mid-session."""
    try:
        import numpy as np
        import pandas as pd
        print(f"numpy={np.__version__} pandas={pd.__version__}")
        return
    except ValueError as exc:
        message = str(exc)
        if "numpy.dtype size changed" not in message:
            raise
        print("Detected numpy/pandas ABI mismatch:", message)
        if not REPAIR_NUMPY_ABI:
            raise RuntimeError(
                "Set REPAIR_NUMPY_ABI = True in this setup cell, rerun it, then restart the kernel."
            ) from exc
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--force-reinstall",
                "numpy<2",
                "pandas",
                "scipy",
                "scikit-learn",
                "matplotlib",
                "seaborn",
            ],
            check=True,
        )
        raise RuntimeError(
            "Repaired numpy/pandas ABI packages. Restart the Colab/VSCode kernel, then rerun with REPAIR_NUMPY_ABI = True or False."
        ) from exc

if INSTALL_MINIMAL_DEPS:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.41",
            "tqdm",
            "pandas",
            "matplotlib",
            "seaborn",
            "scikit-learn",
        ],
        check=True,
    )

ensure_science_stack_abi()

def patch_v4_model_hook_for_new_transformers():
    path = Path('synthetic_niah_v4/model.py')
    if not path.exists():
        print('v4 model hook patch skipped; file not found:', path)
        return
    text = path.read_text(encoding='utf-8')
    old = '''                handles.append(
                    block.register_forward_pre_hook(
                        lambda _module, inputs, layer_idx=layer_idx: (
                            apply(f"resid_pre_layer_{layer_idx}", inputs[0]),
                            *inputs[1:],
                        )
                    )
                )

                def post_hook(_module, _inputs, output, layer_idx=layer_idx):
                    hidden = apply(f"resid_post_layer_{layer_idx}", output[0])
                    return (hidden, *output[1:])
'''
    new = '''                def pre_hook(_module, inputs, layer_idx=layer_idx):
                    if not inputs:
                        return inputs
                    hidden = inputs[0]
                    if isinstance(hidden, tuple):
                        if not hidden:
                            return inputs
                        patched = (apply(f"resid_pre_layer_{layer_idx}", hidden[0]), *hidden[1:])
                        return (patched, *inputs[1:])
                    return (apply(f"resid_pre_layer_{layer_idx}", hidden), *inputs[1:])

                handles.append(
                    block.register_forward_pre_hook(pre_hook)
                )

                def post_hook(_module, _inputs, output, layer_idx=layer_idx):
                    if isinstance(output, tuple):
                        hidden = apply(f"resid_post_layer_{layer_idx}", output[0])
                        return (hidden, *output[1:])
                    return apply(f"resid_post_layer_{layer_idx}", output)
'''
    if old in text:
        path.write_text(text.replace(old, new), encoding='utf-8')
        print('Patched synthetic_niah_v4/model.py for newer Transformers/PyTorch hook outputs.')
    elif 'return apply(f"resid_post_layer_{layer_idx}", output)' in text:
        print('v4 model hook patch already present.')
    else:
        print('v4 model hook patch not applied; source pattern did not match.')

patch_v4_model_hook_for_new_transformers()

def patch_v4_steering_projection_r2_merge():
    path = Path('synthetic_niah_v4/steering.py')
    if not path.exists():
        print('v4 steering patch skipped; file not found:', path)
        return
    text = path.read_text(encoding='utf-8')
    changed = False
    if 'if "projection_r2" not in direction_df.columns:' not in text:
        text = text.replace(
            '    if direction_df.empty:\n        return direction_df\n    sub = direction_df[\n',
            '    if direction_df.empty:\n        return direction_df\n    if "projection_r2" not in direction_df.columns:\n        direction_df = direction_df.copy()\n        direction_df["projection_r2"] = 0.0\n    sub = direction_df[\n',
        )
        changed = True
    if 'def _load_direction_configs(run_dir: Path) -> pd.DataFrame:' not in text:
        helper = '''\n\ndef _load_direction_configs(run_dir: Path) -> pd.DataFrame:\n    artifacts = run_dir / "artifacts"\n    tables = run_dir / "tables"\n    direction_meta_path = artifacts / "directions.csv"\n    direction_metrics_path = tables / "direction_metrics.csv"\n    direction_df = pd.read_csv(direction_meta_path) if direction_meta_path.exists() else pd.DataFrame()\n    if direction_df.empty:\n        return direction_df\n    metrics_df = pd.read_csv(direction_metrics_path) if direction_metrics_path.exists() and direction_metrics_path.stat().st_size > 0 else pd.DataFrame()\n    if metrics_df.empty or "projection_r2" not in metrics_df.columns:\n        direction_df = direction_df.copy()\n        direction_df["projection_r2"] = 0.0\n        direction_df["projection_slope"] = 0.0\n        return direction_df\n    key_cols = ["model_type", "eval_mode", "anchor_name", "anchor_k", "hook_name", "layer", "direction_type", "target"]\n    left = direction_df.copy()\n    right = metrics_df.copy()\n    for col in key_cols:\n        if col not in left.columns:\n            left[col] = ""\n        if col not in right.columns:\n            right[col] = ""\n        left[f"__key_{col}"] = left[col].fillna("").astype(str)\n        right[f"__key_{col}"] = right[col].fillna("").astype(str)\n    merge_cols = [f"__key_{col}" for col in key_cols]\n    keep = merge_cols + ["projection_r2", "projection_slope"]\n    merged = left.merge(right[keep].drop_duplicates(merge_cols), on=merge_cols, how="left")\n    merged["projection_r2"] = merged["projection_r2"].fillna(0.0)\n    merged["projection_slope"] = merged["projection_slope"].fillna(0.0)\n    return merged.drop(columns=merge_cols)\n'''
        text = text.replace('\n\ndef _distribution(preds: list[int]) -> np.ndarray:\n', helper + '\n\ndef _distribution(preds: list[int]) -> np.ndarray:\n')
        changed = True
    old = '    direction_meta_path = artifacts / "directions.csv"\n    direction_npz_path = artifacts / "directions.npz"\n    direction_df = pd.read_csv(direction_meta_path) if direction_meta_path.exists() else pd.DataFrame()\n'
    new = '    direction_npz_path = artifacts / "directions.npz"\n    direction_df = _load_direction_configs(run_dir)\n'
    if old in text:
        text = text.replace(old, new)
        changed = True
    if changed:
        path.write_text(text, encoding='utf-8')
        print('Patched synthetic_niah_v4/steering.py to merge projection_r2 from direction_metrics.csv.')
    else:
        print('v4 steering projection_r2 patch already present.')

patch_v4_steering_projection_r2_merge()



def patch_v4_plot_empty_data_guards():
    path = Path('synthetic_niah_v4/plots.py')
    if not path.exists():
        print('v4 plots empty-data patch skipped; file not found:', path)
        return
    current = path.read_text(encoding='utf-8')
    if '_plot_bar_or_placeholder' in current and '_plot_heatmap_or_placeholder' in current:
        print('v4 plots empty-data patch already present.')
        return
    path.write_text('from __future__ import annotations\n\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport seaborn as sns\n\n\ndef _save(path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    plt.tight_layout()\n    plt.savefig(path, dpi=150, bbox_inches="tight")\n    plt.close()\n\n\ndef _placeholder(path: Path, title: str) -> None:\n    plt.figure(figsize=(6, 3.5))\n    plt.text(0.5, 0.5, "No data", ha="center", va="center")\n    plt.axis("off")\n    plt.title(title)\n    _save(path)\n\n\ndef _read(path: Path) -> pd.DataFrame:\n    return pd.read_csv(path) if path.exists() and path.stat().st_size > 0 else pd.DataFrame()\n\n\ndef _has_columns(df: pd.DataFrame, columns: list[str]) -> bool:\n    return not df.empty and all(col in df.columns for col in columns)\n\n\ndef _clean_numeric(df: pd.DataFrame, numeric_cols: list[str]) -> pd.DataFrame:\n    if df.empty:\n        return df.copy()\n    out = df.copy()\n    for col in numeric_cols:\n        if col in out.columns:\n            out[col] = pd.to_numeric(out[col], errors="coerce")\n    return out\n\n\ndef _has_finite(series: pd.Series) -> bool:\n    values = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)\n    return bool(np.isfinite(values).any())\n\n\ndef _plot_bar_or_placeholder(\n    df: pd.DataFrame,\n    path: Path,\n    title: str,\n    x: str,\n    y: str,\n    hue: str | None = None,\n    ylim: tuple[float, float] | None = None,\n) -> None:\n    needed = [x, y] + ([hue] if hue else [])\n    if not _has_columns(df, needed):\n        _placeholder(path, title)\n        return\n    clean = _clean_numeric(df, [y]).dropna(subset=[x, y])\n    if clean.empty or not _has_finite(clean[y]):\n        _placeholder(path, title)\n        return\n    plt.figure(figsize=(10, 5))\n    sns.barplot(data=clean, x=x, y=y, hue=hue, errorbar=None)\n    plt.xticks(rotation=35, ha="right")\n    if ylim is not None:\n        plt.ylim(*ylim)\n    plt.title(title)\n    _save(path)\n\n\ndef _plot_heatmap_or_placeholder(\n    mat: pd.DataFrame,\n    path: Path,\n    title: str,\n    fmt: str = ".2f",\n    center: float | None = None,\n) -> None:\n    if mat.empty:\n        _placeholder(path, title)\n        return\n    mat = mat.apply(pd.to_numeric, errors="coerce")\n    if not np.isfinite(mat.to_numpy(dtype=float)).any():\n        _placeholder(path, title)\n        return\n    plt.figure(figsize=(8, 5))\n    sns.heatmap(mat, annot=True, fmt=fmt, cmap="vlag", center=center)\n    plt.title(title)\n    _save(path)\n\n\ndef make_probe_plots(run_dir: Path) -> None:\n    figures = run_dir / "figures"\n    probe = _read(run_dir / "tables" / "probe_results.csv")\n    if probe.empty:\n        _placeholder(figures / "probe_acc_by_layer_anchor.png", "Probe accuracy by layer/anchor")\n        _placeholder(figures / "probe_r2_by_layer_anchor.png", "Probe R2 by layer/anchor")\n        _placeholder(figures / "probe_minus_baseline_heatmap.png", "Probe minus baseline")\n        return\n    raw = probe[probe["raw_or_residualized"].eq("raw")]\n    acc = raw[raw["probe_type"].eq("multiclass_logistic")]\n    if acc.empty:\n        acc = raw\n    _plot_bar_or_placeholder(\n        acc,\n        figures / "probe_acc_by_layer_anchor.png",\n        "v4 probe accuracy by layer and anchor",\n        x="anchor_name",\n        y="accuracy",\n        hue="layer",\n        ylim=(0, 1),\n    )\n\n    ridge = raw[raw["probe_type"].eq("ridge_scalar")]\n    if ridge.empty:\n        ridge = raw\n    _plot_bar_or_placeholder(\n        ridge,\n        figures / "probe_r2_by_layer_anchor.png",\n        "v4 ridge R2 by layer and anchor",\n        x="anchor_name",\n        y="r2",\n        hue="layer",\n    )\n\n    if _has_columns(acc, ["anchor_name", "layer", "accuracy", "position_baseline_acc"]):\n        acc = _clean_numeric(acc, ["accuracy", "position_baseline_acc"])\n        acc["minus_position"] = acc["accuracy"] - acc["position_baseline_acc"]\n        mat = acc.pivot_table(index="anchor_name", columns="layer", values="minus_position", aggfunc="mean")\n    else:\n        mat = pd.DataFrame()\n    _plot_heatmap_or_placeholder(\n        mat,\n        figures / "probe_minus_baseline_heatmap.png",\n        "Probe accuracy minus position baseline",\n        center=0.0,\n    )\n\n\ndef make_direction_plots(run_dir: Path) -> None:\n    figures = run_dir / "figures"\n    directions = _read(run_dir / "tables" / "direction_metrics.csv")\n    if directions.empty:\n        _placeholder(figures / "direction_cosine_heatmap.png", "Direction cosines")\n        _placeholder(figures / "projection_by_count.png", "Projection by count")\n        _placeholder(figures / "input_geometry_projection_trajectories.png", "Input geometry")\n        return\n    if _has_columns(directions, ["direction_type", "anchor_name", "cosine_with_unembedding"]):\n        mat = directions.pivot_table(index="direction_type", columns="anchor_name", values="cosine_with_unembedding", aggfunc="mean")\n    else:\n        mat = pd.DataFrame()\n    _plot_heatmap_or_placeholder(\n        mat,\n        figures / "direction_cosine_heatmap.png",\n        "Cosine with unembedding adjacent direction",\n        center=0.0,\n    )\n\n    if _has_columns(directions, ["projection_r2", "anchor_name", "projection_slope", "direction_type"]):\n        top = _clean_numeric(directions, ["projection_r2", "projection_slope"]).dropna(subset=["projection_r2"]).sort_values("projection_r2", ascending=False).head(24)\n    else:\n        top = pd.DataFrame()\n    _plot_bar_or_placeholder(\n        top,\n        figures / "projection_by_count.png",\n        "Projection slope by anchor/direction",\n        x="anchor_name",\n        y="projection_slope",\n        hue="direction_type",\n    )\n\n    geom = _read(run_dir / "tables" / "input_geometry_results.csv")\n    _plot_bar_or_placeholder(\n        geom,\n        figures / "input_geometry_projection_trajectories.png",\n        "Input geometry projection summary",\n        x="anchor_name",\n        y="projection_slope",\n        hue="direction_type",\n    )\n\n\ndef make_steering_plots(run_dir: Path) -> None:\n    figures = run_dir / "figures"\n    steering = _read(run_dir / "tables" / "steering_results.csv")\n    if steering.empty:\n        _placeholder(figures / "steering_heatmap_anchor_layer.png", "Steering heatmap")\n        _placeholder(figures / "steering_dose_response_top_configs.png", "Steering dose response")\n        _placeholder(figures / "steering_controls.png", "Steering controls")\n        return\n    main = steering[steering["control_type"].eq("none")]\n    if _has_columns(main, ["anchor_name", "layer", "mean_count_shift"]):\n        mat = main.pivot_table(index="anchor_name", columns="layer", values="mean_count_shift", aggfunc="mean")\n    else:\n        mat = pd.DataFrame()\n    _plot_heatmap_or_placeholder(\n        mat,\n        figures / "steering_heatmap_anchor_layer.png",\n        "Mean count shift by anchor/layer",\n        center=0.0,\n    )\n\n    if _has_columns(main, ["model_type", "anchor_name", "layer", "direction_type", "mean_count_shift", "alpha", "mean_pred_steered"]):\n        main_clean = _clean_numeric(main, ["mean_count_shift", "alpha", "mean_pred_steered"])\n        top_keys = (\n            main_clean.groupby(["model_type", "anchor_name", "layer", "direction_type"], as_index=False)["mean_count_shift"]\n            .apply(lambda s: s.abs().max())\n            .rename(columns={"mean_count_shift": "abs_shift"})\n            .dropna(subset=["abs_shift"])\n            .sort_values("abs_shift", ascending=False)\n            .head(6)\n        )\n        merged = main_clean.merge(top_keys[["model_type", "anchor_name", "layer", "direction_type"]]) if not top_keys.empty else pd.DataFrame()\n    else:\n        merged = pd.DataFrame()\n    if merged.empty or not _has_finite(merged["mean_pred_steered"]):\n        _placeholder(figures / "steering_dose_response_top_configs.png", "Top steering dose response")\n    else:\n        plt.figure(figsize=(9, 5))\n        sns.lineplot(data=merged, x="alpha", y="mean_pred_steered", hue="anchor_name", style="direction_type", marker="o", errorbar=None)\n        plt.title("Top steering dose response")\n        _save(figures / "steering_dose_response_top_configs.png")\n\n    _plot_bar_or_placeholder(\n        steering,\n        figures / "steering_controls.png",\n        "Steering controls",\n        x="control_type",\n        y="mean_count_shift",\n        hue="direction_type",\n    )\n\n\ndef make_patching_plots(run_dir: Path) -> None:\n    figures = run_dir / "figures"\n    patch = _read(run_dir / "tables" / "interchange_patching_results.csv")\n    if patch.empty:\n        _placeholder(figures / "interchange_patch_matrix.png", "Interchange patch matrix")\n        return\n    if _has_columns(patch, ["receiver_count", "donor_count", "causal_effect_size"]):\n        mat = patch.pivot_table(index="receiver_count", columns="donor_count", values="causal_effect_size", aggfunc="mean")\n    else:\n        mat = pd.DataFrame()\n    _plot_heatmap_or_placeholder(\n        mat,\n        figures / "interchange_patch_matrix.png",\n        "Patched pred shift by receiver/donor count",\n        fmt=".1f",\n        center=0.0,\n    )\n\n\ndef make_all_plots(run_dir: Path) -> None:\n    sns.set_theme(style="whitegrid", context="notebook")\n    make_probe_plots(run_dir)\n    make_direction_plots(run_dir)\n    make_steering_plots(run_dir)\n    make_patching_plots(run_dir)\n', encoding='utf-8')
    print('Patched synthetic_niah_v4/plots.py to skip empty/all-NaN plot inputs.')

patch_v4_plot_empty_data_guards()

if INSTALL_EDITABLE_PACKAGE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)
print('cwd =', Path.cwd())

## Runtime settings

- `debug`: quick end-to-end sanity check.
- `main`: v2-style main setting: two models, one seed by default, `1..10` counts, GPT-2 learned absolute positions.
- `stage='all'` runs training, behavior eval, hidden cache, probes, direction extraction, patching, steering, plots, and a markdown/html artifact report.

In [2]:
PRESET = 'debug'  # 'debug' or 'main'
STAGE = 'all'
SEEDS = '1234'
OUT_ROOT = 'runs/synthetic_niah_v4'
RUN_NAME = ''
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
SKIP_COMPLETED = True

print({
    'PRESET': PRESET,
    'STAGE': STAGE,
    'SEEDS': SEEDS,
    'OUT_ROOT': OUT_ROOT,
    'DEVICE': DEVICE,
    'SKIP_COMPLETED': SKIP_COMPLETED,
})

{'PRESET': 'debug', 'STAGE': 'all', 'SEEDS': '1234', 'OUT_ROOT': 'runs/synthetic_niah_v4', 'DEVICE': 'cuda', 'SKIP_COMPLETED': True}


In [ ]:
cmd = [
    sys.executable, '-u', '-m', 'synthetic_niah_v4.run_v4',
    '--preset', PRESET,
    '--stage', STAGE,
    '--device', DEVICE,
    '--out-root', OUT_ROOT,
    '--seeds', SEEDS,
]
if RUN_NAME:
    cmd += ['--run-name', RUN_NAME]
if SKIP_COMPLETED:
    cmd.append('--skip-completed')
print(' '.join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
captured = []
for line in proc.stdout:
    print(line, end='')
    captured.append(line.rstrip())
returncode = proc.wait()
if returncode:
    print('---- Last 120 log lines ----')
    print('\n'.join(captured[-120:]))
    raise subprocess.CalledProcessError(returncode, cmd)

RUN_DIR = None
for line in reversed(captured):
    if line.startswith('FINAL_RUN_DIR '):
        RUN_DIR = Path(line.split(' ', 1)[1].strip())
        break
if RUN_DIR is None:
    RUN_DIR = Path(OUT_ROOT) / RUN_NAME if RUN_NAME else Path(OUT_ROOT)
print('RUN_DIR =', RUN_DIR)

## Result tables

The most important tables are behavior accuracy, probe results, direction diagnostics, and steering/patching. Probe accuracy alone means decodability; steering or patching is the causal check.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

tables = RUN_DIR / 'tables'
for name in [
    'behavior_accuracy_by_count.csv',
    'probe_results.csv',
    'direction_metrics.csv',
    'steering_results.csv',
    'interchange_patching_results.csv',
]:
    path = tables / name
    display(Markdown(f'### `{name}`'))
    if path.exists() and path.stat().st_size > 0:
        display(pd.read_csv(path).head(20))
    else:
        display(Markdown('_No rows._'))

## Figures

These figures summarize: probe accuracy/R2 by anchor and layer, direction agreement with the count-token unembedding, steering dose response, steering controls, and patching from donor to receiver counts.

In [ ]:
from IPython.display import Image

for fig in sorted((RUN_DIR / 'figures').glob('*.png')):
    display(Markdown(f'### `{fig.name}`'))
    display(Image(filename=str(fig), width=900))

## Save to Google Drive

In [ ]:
SAVE_TO_DRIVE = False
DRIVE_SAVE_COMPLETED = False
DRIVE_DIR = '/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/Synthetic_CoT_NiaH_Count/colab_results/'

if SAVE_TO_DRIVE and Path('/content').exists():
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    dest = Path(DRIVE_DIR) / RUN_DIR.name
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(RUN_DIR, dest)
    DRIVE_SAVE_COMPLETED = True
    print('saved to', dest)
else:
    print('SAVE_TO_DRIVE is False; local RUN_DIR:', RUN_DIR)

## Auto-disconnect Colab Runtime

This cell runs immediately after the Google Drive save cell. It only disconnects when a Drive save was confirmed; local VSCode/Jupyter runs are not force-closed by default.

In [ ]:
AUTO_DISCONNECT_AFTER_DRIVE_SAVE = True
FORCE_LOCAL_KERNEL_SHUTDOWN = False

if AUTO_DISCONNECT_AFTER_DRIVE_SAVE and globals().get("DRIVE_SAVE_COMPLETED", False):
    import time

    print("Google Drive save completed. Flushing Drive and disconnecting Colab runtime in 3 seconds...")
    time.sleep(3)
    try:
        from google.colab import drive, runtime

        try:
            drive.flush_and_unmount()
            print("Google Drive flushed and unmounted.")
        except Exception as e:
            print(f"Drive flush/unmount skipped or failed: {e}")
        runtime.unassign()
    except Exception as e:
        print(f"Colab runtime disconnect unavailable or failed: {e}")
        if FORCE_LOCAL_KERNEL_SHUTDOWN:
            import IPython

            IPython.Application.instance().kernel.do_shutdown(restart=False)
        else:
            print("Not forcing local kernel shutdown.")
else:
    print("Auto-disconnect skipped: no confirmed Google Drive save, or AUTO_DISCONNECT_AFTER_DRIVE_SAVE is False.")

## Optional GitHub result upload

In [ ]:
PUSH_RESULTS_TO_GITHUB = False
GIT_BRANCH = 'v4-results'

if PUSH_RESULTS_TO_GITHUB:
    subprocess.run(['git', 'checkout', '-B', GIT_BRANCH], check=True)
    subprocess.run(['git', 'add', str(RUN_DIR)], check=True)
    subprocess.run(['git', 'commit', '-m', f'Add v4 results {RUN_DIR.name}'], check=False)
    subprocess.run(['git', 'push', '-u', 'origin', GIT_BRANCH], check=True)
else:
    print('PUSH_RESULTS_TO_GITHUB is False')